# TP Jour 1 — Choix des Datasets Kaggle (Mono ou Multi-Sources), HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et de combiner **un dataset multi-tables ou plusieurs datasets publics complémentaires sur Kaggle** afin de les croiser par jointure.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner **un ou plusieurs datasets Kaggle** (~50 à 500 Mo au total) partageant au moins un axe de jointure (date/heure, zone géographique, code ISO/catégorie, identifiant métier).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts multi-sources sur HDFS** (`/lakehouse/bronze/raw/<source>/...`) et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** avec traçabilité de la source (`_source_dataset`, `_source_file`, `_ingested_at`) et inspecter vos jobs sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de votre projet *(À compléter par le groupe)*
- **Titre du sujet :** Retards des vols intérieurs américains en 2015 : quels vols sont à risque ?
- **Dataset(s) Kaggle retenu(s) (`auteur/nom-du-dataset`) :**
  1. Source principale : `usdot/flight-delays`, table `flights.csv` (5,8 millions de vols, 565 Mo)
  2. Source(s) complémentaire(s) pour jointure : les deux référentiels du même dataset, `airlines.csv` (14 compagnies) et `airports.csv` (322 aéroports avec ville, État et coordonnées)
- **Clé(s) de jointure envisagée(s) entre sources / tables :** code IATA de la compagnie (`airline`) et code IATA de l'aéroport, utilisé deux fois (`origin_airport` et `destination_airport`)
- **Problématique métier principale :** qu'est-ce qui fait qu'un vol arrive en retard, et peut-on repérer à l'avance, au moment de réserver, les vols à risque ?
- **Variables clés retenues (temporelles, quantitatives, catégorielles) :**
  - temporelles : date du vol (`year`, `month`, `day`, `day_of_week`), heure de départ prévue (`scheduled_departure`) ;
  - quantitatives : retard à l'arrivée (`arrival_delay`, variable principale), retard au départ, temps de roulage (`taxi_out`), distance, minutes de retard par cause (système aérien, météo, compagnie, avion arrivé en retard, sécurité) ;
  - catégorielles : compagnie, aéroports de départ et d'arrivée, annulation et son motif.

## 1. Téléchargement de vos Datasets Kaggle dans `/workspace/data_raw/<source>/`

Renseignez dans la liste `KAGGLE_DATASETS` un ou plusieurs identifiants Kaggle (ex. `["auteur1/dataset-principal", "auteur2/meteo-ou-referentiel"]`), ou déposez directement vos fichiers/sous-dossiers dans `data_raw/`.
- Chaque dataset est isolé dans son propre sous-dossier `data_raw/<nom-dataset>/` pour éviter toute collision de noms (`train.csv`, `metadata.csv`, etc.) entre plusieurs sources.
- **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [1]:
import re
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# Un seul dataset Kaggle, multi-tables : flights (faits) + airlines et airports (référentiels pour les jointures)
KAGGLE_DATASETS: list[str] = [
    "usdot/flight-delays",
]

for slug in KAGGLE_DATASETS:
    slug = slug.strip()
    if not slug:
        continue
    subdir = re.sub(r"[^a-zA-Z0-9._-]+", "_", slug.split("/")[-1]).strip("_") or "dataset"
    target_dir = DATA_RAW / subdir
    existing = [f for f in target_dir.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS] if target_dir.exists() else []
    if not existing:
        import kagglehub
        print(f"Téléchargement de {slug} -> {target_dir}/ ...")
        cache_dir = Path(kagglehub.dataset_download(slug))
        target_dir.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(cache_dir.rglob("*")):
            if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
                dest_file = target_dir / src_file.relative_to(cache_dir)
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_file, dest_file)

local_files = sorted([f for f in DATA_RAW.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, (
    "Aucun fichier détecté dans /workspace/data_raw/. "
    "Renseignez `KAGGLE_DATASETS` ci-dessus ou utilisez `task data:download DATASET='auteur1/ds1,auteur2/ds2'`."
)

print(f"Inventaire local multi-sources dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    rel = f.relative_to(DATA_RAW)
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {str(rel):<55} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

Inventaire local multi-sources dans /workspace/data_raw (3 fichier(s)) :
 - flight-delays/airlines.csv                                 0.00 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - flight-delays/airports.csv                                 0.02 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - flight-delays/flights.csv                                564.96 Mo  (~36 bloc(s) HDFS de 16 Mo)


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J1_HDFS_Ingestion_Bronze")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .getOrCreate()
)

assert spark is not None, "TODO : Initialisez `spark` avec SparkSession.builder..."

# Les 3 paramètres HDFS lus dans la configuration Spark
default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7e76d077-84c7-47f2-8d05-968767dd919c;1.0
	confs: [default]


	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central


	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central


	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.4.1 in central


	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central


	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.7-3 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found commons-logging#commons-logging;1.3.0 in central
	found org.apache.parquet#parquet-jackson;1.16.0 in central
	found io.delta#delta-kernel-unitycatalog;4.4.1 in central
:: resolution report :: resolve 1162ms :: artifacts dl 20ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-core;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-databind;2.13.5 from central in [default]
	com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 from central in [default]
	com.github.luben#zstd-jni;1.5.7-3 from central in [default]
	com.google.code.findbugs#jsr305;3.0.2 from central in [default]
	commons-logging#commons-logging;1.3.0 from central in [default]
	commons-pool#commons-poo

26/11/19 13:01:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/opt/venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version       : 4.2.0
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [4]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/flight-delays/airlines.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/flight-delays/airports.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/flight-delays/flights.csv


In [5]:
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : flight-delays/flights.csv (564.96 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #5 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #6 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #7 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #8 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #9 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #10 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #11 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #12 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #13 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #14 (16.00 Mo) -> DataNodes : ['dat

### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

**Réponses**

1. `flights.csv` fait 564,96 Mo, soit 564,96 / 16 = 35,3 blocs. Le NameNode en liste bien **36** (cellule précédente) : 35 blocs pleins de 16 Mo et un dernier de 4,96 Mo. Les deux petits fichiers (`airports.csv`, `airlines.csv`) tiennent chacun dans un seul bloc.
2. **Non.** 16 Mo est la taille maximale d'un bloc, pas une taille réservée. Un fichier de 3 Mo forme un bloc de 3 Mo, stocké comme un fichier de 3 Mo sur le disque de chaque DataNode qui en a une copie (donc 6 Mo au total avec 2 copies). Le vrai coût d'un petit fichier est ailleurs : le NameNode garde en mémoire une entrée par fichier et par bloc, quelle que soit leur taille. Beaucoup de petits fichiers saturent donc la mémoire du NameNode, pas les disques (d'où l'`OPTIMIZE` de J3).
3. Avec 2 copies, on peut perdre un DataNode sans perdre de données. Le 3e DataNode sert de **place libre** : quand un nœud tombe, le NameNode y recopie les blocs qui n'ont plus qu'une copie, et le facteur 2 est rétabli. Avec 3 copies sur 3 nœuds, une panne laisserait les blocs sous-répliqués sans aucun nœud pour les reconstruire. 2 copies coûtent aussi moins de disque que 3 (2 fois la taille des données au lieu de 3).

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [4]:
from collections import Counter

print("Fichier cible du test de résilience :", largest_hdfs_path)

# État des DataNodes vu par le NameNode au moment de la lecture
RT = jvm.org.apache.hadoop.hdfs.protocol.HdfsConstants.DatanodeReportType
vivants = sorted(d.getHostName() for d in fs.getDataNodeStats(RT.LIVE))
morts = sorted(d.getHostName() for d in fs.getDataNodeStats(RT.DEAD))
print("DataNodes vivants :", vivants, "| DataNodes morts :", morts)

# Où sont les copies des blocs du fichier, et combien n'ont plus qu'une copie sur un nœud vivant
status_check = fs.getFileStatus(PathHDFS(largest_hdfs_path))
blocs = fs.getFileBlockLocations(status_check, 0, status_check.getLen())
copies = Counter(h for b in blocs for h in b.getHosts())
sous_repliques = sum(1 for b in blocs if len([h for h in b.getHosts() if h in vivants]) < 2)
print(f"{len(blocs)} blocs | copies par DataNode : {dict(sorted(copies.items()))} | "
      f"blocs avec moins de 2 copies vivantes : {sous_repliques}")

# Lecture complète du fichier : elle doit réussir même avec un DataNode arrêté
df_check = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(largest_hdfs_path)
)

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")
print(f"Cœurs Spark (defaultParallelism)  : {spark.sparkContext.defaultParallelism}")

Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/flight-delays/flights.csv
DataNodes vivants : ['datanode1', 'datanode2', 'datanode3'] | DataNodes morts : []
36 blocs | copies par DataNode : {'datanode1': 36, 'datanode3': 36} | blocs avec moins de 2 copies vivantes : 0


Nombre total de lignes lues       : 5,819,079
Nombre de partitions Spark créées : 8
Cœurs Spark (defaultParallelism)  : 8


**Partitions Spark et blocs HDFS**

Le fichier a 36 blocs mais Spark le lit en **8 partitions** : il n'y a pas une partition par bloc. Pour les fichiers, Spark fixe la taille d'une partition au minimum entre 128 Mo (`spark.sql.files.maxPartitionBytes`) et la taille totale divisée par le nombre de cœurs : 565 Mo / 8 cœurs ≈ 71 Mo. Chaque partition regroupe donc 4 à 5 blocs de 16 Mo, et les 8 cœurs travaillent en même temps. Avec des blocs de 128 Mo (valeur par défaut de HDFS), on retrouverait à peu près une partition par bloc.

**Crash-test**

- La lecture se fait en continu pendant la panne : le client HDFS demande au NameNode où sont les blocs, puis va chercher chaque bloc sur un DataNode vivant qui en a une copie.
- Environ 50 secondes après l'arrêt, le NameNode déclare le nœud mort. Les blocs qui n'ont plus qu'une copie sont recopiés sur un nœud vivant (sur l'interface du NameNode, le nombre de blocs sous-répliqués revient à 0).
- Quand le nœud redémarre, certains blocs ont 3 copies : le NameNode supprime la copie en trop.
- **Ce qu'on a observé** : pendant la panne de `datanode2`, la lecture a renvoyé les 5 819 079 lignes sans erreur. On en voit encore la trace : toutes les copies des blocs Bronze sont aujourd'hui sur `datanode1` et `datanode3` (inventaire des blocs ci-dessus). Les blocs de `datanode2` ont été recopiés sur les deux autres nœuds pendant la panne, puis ses copies devenues en trop ont été supprimées à son retour.
- La cellule ci-dessus a été réexécutée après le test, d'où les 3 DataNodes vivants.

---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de vos datasets dans `/lakehouse/bronze/<nom_table>`.

In [ ]:
for f in local_files:
    rel = f.relative_to(DATA_RAW)
    dataset_name = rel.parts[0] if len(rel.parts) > 1 else "default"
    table_name = f.stem.lower().replace(" ", "_")
    raw_path = f"/lakehouse/bronze/raw/{rel.as_posix()}"
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    df_raw = (
        spark.read.parquet(raw_path)
        if f.suffix.lower() == ".parquet"
        else spark.read.option("header", "true").option("inferSchema", "true").option("multiLine", "true").option("escape", '"').csv(raw_path)
    )
    df_raw = (
        df_raw.toDF(*[c.strip().lower().replace(" ", "_") for c in df_raw.columns])
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_dataset", F.lit(dataset_name))
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    df_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_delta_path)
    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path:<55} (source={dataset_name}) -> {cnt:>10,} lignes")

26/11/17 10:40:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[BRONZE OK] /lakehouse/bronze/airlines                              (source=flight-delays) ->         14 lignes


[BRONZE OK] /lakehouse/bronze/airports                              (source=flight-delays) ->        322 lignes


[BRONZE OK] /lakehouse/bronze/flights                               (source=flight-delays) ->  5,819,079 lignes


In [8]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['airlines', 'airports', 'flights']
